In [ ]:
import importlib
import sys
import numpy as np
import re
import math

spec = importlib.util.spec_from_file_location("lib", "lib/__init__.py")
module_obj = importlib.util.module_from_spec(spec)
sys.modules["lib"] = module_obj
spec.loader.exec_module(module_obj)

from lib import PipelineConfig

In [3]:
if "snakemake" in locals():
    feeder_outputs = snakemake.params.feeder_outputs
    no_feeder_outputs = snakemake.params.no_feeder_outputs
    input_paths = snakemake.input
    pipeline_config = PipelineConfig(snakemake.params.config, **snakemake.params.pipeline_kwargs)
    scenario_name = snakemake.wildcards.scenario
    scenario_config = pipeline_config.scenarios[scenario_name]
else:
    raise Exception("This notebook is only snakemake-compatible for now")

Exception: This notebook is only snakemake-compatible for now

In [ ]:
print("Values considered for the radius")
scenario_config.feeders.radiis

In [ ]:
print("Values considered for the frequency")
scenario_config.feeders.frequencies

In [ ]:
print("Values considered for the speeed")
scenario_config.feeders.speeds

In [ ]:
print("List of passed input paths")
feeder_outputs

Each Path is of the form /path/to/something/simulated_{scenario}_transitWithAbstractAccess_{radius}_{frequency}_{speed_index}
- radius: the value of the radius parameter
- frequency: value of the frequency
- speed_index: index of the speed value in the speeds list displayed above (I didn't want to risk passing floats around)

Next we display the path to the outputs of the simulation corresponding to the scenario without feeder service
The variable can be none if no such simulation is provided

In [ ]:
no_feeder_outputs

# What to do next ?

In general we want to measure the relationship between the level of introduction of feeder services and how much better could the trips get relatively to existing public transport.

Some sanity checks, verify that all the scenarios have the same set of trips (person_id, person_trip_id, departure_time, origin, destination). If not, keep only the same ones while we investigate the source of the problem.

Parse the simulation outputs corresponding to each feeder settings and do some analysis.
Maybe the interesting KPI to consider here is the route cost of each trip. It is the one optimized directly by the routing algorithm so I think it makes sense. Here are the parameters:
- `rail_u_h`, `subway_u_h`, `bus_u_h`, `tram_u_h`, `other_u_h`: the marginal utility of time (in hours) spent in respective PT modes. The default value of these parameters is -7.
- `wait_u_h`: the marginal utility of time (in hours) spent waiting for public transport, no matter its mode.  The default value of this parameter is -6.
- `walk_u_h`: the markinal utility of time (in hours) spent walking to/from/between public transport legs.  Its default value is -7.
- `transfer_u`: the marginal utility of a transfer. Its default value is -1.

We can consider the whole chain including feeders (so also considering those transfers) and weighing time spent in a feeder service similarly to the bus. Instead of just keeping the total cost for the trip, we can also have the different components separated.

Then a sanity check we can have is to make sure that no trip sees its routing cost get worse before/after feeder.

Then we can have various graphs with this
- One line plot, feeder radius on the x-axis, total cost on the y axis. If multiple frequencies are considered, we put them in differently coloured lines. If multiple speeds are considered, we can used dashed lines.
- Same plot but instead of total routing cost, we can have the number of trips where the routing cost improves.
- Stacked bar plot, x axis for the feeder radius, y axis for the total cost, colors for the cost components, facet_col for frequency and facet_row for speed.
Then we can check the impact on the usage of existing PT systems (even though we should already kind of have an idea with the cost components)
- A plot showing the number of pt legs per pt mode (rail, subway, tram, feeder) in each setting.
- ...Other analyses

In [ ]:
import pandas as pd
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Load data

## No feeder

no_feeder_outputs = no_feeder_outputs[0]

nofeeder_legs = pd.read_csv(os.path.join(no_feeder_outputs, "eqasim_legs.csv"), sep=";")
nofeeder_pt = pd.read_csv(os.path.join(no_feeder_outputs, "eqasim_pt.csv"), sep=";")
nofeeder_trips = pd.read_csv(os.path.join(no_feeder_outputs, "eqasim_trips.csv"), sep=";")
nofeeder_routing_costs = pd.read_csv(os.path.join(no_feeder_outputs, "pt_routing_costs.csv"), sep=";")

# Feeders 

feeders_legs = [pd.read_csv(os.path.join(feeder_output, "eqasim_legs.csv"), sep=";") for feeder_output in feeder_outputs]
feeders_pt = [pd.read_csv(os.path.join(feeder_output, "eqasim_pt.csv"), sep=";") for feeder_output in feeder_outputs]
feeders_trips = [pd.read_csv(os.path.join(feeder_output, "eqasim_trips.csv"), sep=";") for feeder_output in feeder_outputs]
feeders_abstract_access = [pd.read_csv(os.path.join(feeder_output, "eqasim_abstract_access_legs.csv"), sep=";") for feeder_output in feeder_outputs]
feeders_routing_costs = [pd.read_csv(os.path.join(feeder_output, "pt_routing_costs.csv"), sep=";") for feeder_output in feeder_outputs]

nofeeder_legs.head(5)

In [ ]:
## Extract parameters from file paths
def extract_parameters(path):
    """Extraire les paramètres des chemins de fichiers de scénarios feeder
    
    Pattern: simulated_{scenario}_transitWithAbstractAccess_{radius}_{frequency}_{speed_index}
    
    Args:
        path (str): Chemin vers le fichier de sortie du scénario
        
    Returns:
        tuple: (radius, frequency, speed, speed_index)
    """
    pattern = r'simulated_.*_transitWithAbstractAccess_(\d+)_(\d+)_(\d+)'
    match = re.search(pattern, path)
    
    if match:
        radius = int(match.group(1))
        frequency = int(match.group(2))
        speed_index = int(match.group(3))
        speed = scenario_config.feeders.speeds[speed_index]
        return radius, frequency, speed, speed_index
    else:
        print(f"Attention: Impossible d'extraire les paramètres de {path}")
        return None, None, None, None

In [ ]:
print(f"Données chargées:")
print(f"- Scénario sans feeder: {len(nofeeder_routing_costs)} entrées de coût de routage")
print(f"- Scénarios avec feeder: {len(feeders_routing_costs)} scénarios")
for i, feeder_data in enumerate(feeders_routing_costs):
    print(f"  Scénario {i}: {len(feeder_data)} entrées de coût de routage")

In [ ]:
print("Colonnes disponibles dans nofeeder_routing_costs:")
print(list(nofeeder_routing_costs.columns))
print(f"Forme: {nofeeder_routing_costs.shape}")

print("\nAperçu des premières lignes:")
print(nofeeder_routing_costs.head())

print("\nTypes de données:")
print(nofeeder_routing_costs.dtypes)

if len(feeders_routing_costs) > 0:
    print(f"\nColonnes disponibles dans le premier scénario feeder:")
    print(list(feeders_routing_costs[0].columns))
    print(f"Forme: {feeders_routing_costs[0].shape}")

cost_columns = [col for col in nofeeder_routing_costs.columns if 'cost' in col.lower()]
print(f"\nColonnes contenant 'cost': {cost_columns}")

In [ ]:
# Sanity Check

key_columns = ["person_id", "person_trip_id"]
comparison_columns = ["origin_x", "origin_y", "destination_x", "destination_y", "departure_time"]
analysis_columns = key_columns + comparison_columns + ["travel_time"]

summary = []

for i, feeder_trips in enumerate(feeders_trips):
    merged = nofeeder_trips[analysis_columns].merge(
        feeder_trips[analysis_columns],
        on=key_columns,
        suffixes=("_nofeeder", "_feeder"),
        how="outer",
        indicator=True
    )
    only_nofeeder = (merged['_merge'] == 'left_only').sum()
    only_feeder = (merged['_merge'] == 'right_only').sum()
    both = (merged['_merge'] == 'both').sum()
    pct_common = both / len(merged) * 100 if len(merged) > 0 else 0

    common = merged[merged['_merge'] == 'both']
    diffs = {}
    for col in comparison_columns:
        col_nofeeder = f"{col}_nofeeder"
        col_feeder = f"{col}_feeder"
        if col_nofeeder in common and col_feeder in common:
            mask = (~common[col_nofeeder].isna()) & (~common[col_feeder].isna())
            if col in ['origin_x', 'origin_y', 'destination_x', 'destination_y']:
                diff = (abs(common.loc[mask, col_nofeeder] - common.loc[mask, col_feeder]) > 1e-6).sum()
            else:
                diff = (common.loc[mask, col_nofeeder] != common.loc[mask, col_feeder]).sum()
            diffs[col] = diff

    tt_stats = {}
    if "travel_time_nofeeder" in common and "travel_time_feeder" in common:
        mask = (~common["travel_time_nofeeder"].isna()) & (~common["travel_time_feeder"].isna())
        tt_diff = common.loc[mask, "travel_time_feeder"] - common.loc[mask, "travel_time_nofeeder"]
        if len(tt_diff) > 0:
            tt_stats = {
                "mean": tt_diff.mean(),
                "std": tt_diff.std(),
                "min": tt_diff.min(),
                "max": tt_diff.max()
            }
    else:
        print("wtf are we doing here")

    summary.append({
        "scenario": i,
        "total": len(merged),
        "common": both,
        "only_nofeeder": only_nofeeder,
        "only_feeder": only_feeder,
        "pct_common": pct_common,
        "diffs": diffs,
        "tt_stats": tt_stats
    })

for s in summary:
    print(f"Scenario {s['scenario']}: {s['common']}/{s['total']} commun trips ({s['pct_common']:.1f}%)")
    if s['diffs']:
        diff_str = ", ".join(f"{k}:{v}" for k, v in s['diffs'].items() if v > 0)
        if diff_str:
            print(f"    Differences: {diff_str}")
    if s['tt_stats']:
        print(f"    Delta travel_time (s): moy={s['tt_stats']['mean']:.2f}, std={s['tt_stats']['std']:.2f}, min={s['tt_stats']['min']:.2f}, max={s['tt_stats']['max']:.2f}")

df_summary = pd.DataFrame([{
    "Scenario": s["scenario"],
    "Common Trips": s["common"],
    "Only in NoFeeder": s["only_nofeeder"],
    "Only in Feeder": s["only_feeder"]
} for s in summary])

n_cols = 2
n_rows = math.ceil(len(summary) / n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    specs=[[{'type':'domain'}]*n_cols for _ in range(n_rows)],
    subplot_titles=[f"Scenario {s['scenario']}" for s in summary],
)

for i, s in enumerate(summary):
    row = i // n_cols + 1
    col = i % n_cols + 1
    values = [s["common"], s["only_nofeeder"], s["only_feeder"]]
    labels = ["Common Trips", "Only in NoFeeder", "Only in Feeder"]
    fig.add_trace(
        go.Pie(
            labels=labels,
            values=values,
            name=f"Scenario {s['scenario']}",
            textinfo='percent',
            textposition='inside',
            insidetextorientation='radial'
        ),
        row=row, col=col
    )

fig.update_layout(
    title_text="Sanity Check - Répartition des Trips par Scénario",
    height=300*n_rows,
    width=700,
    showlegend=True
)

fig.show()


In [ ]:
def identify_cost_column(df):
    priority_columns = ['total_cost', 'routingCost', 'cost', 'totalCost', 'routing_cost']
    for col in priority_columns:
        if col in df.columns:
            return col
    cost_columns = [col for col in df.columns if 'cost' in col.lower()]
    if cost_columns:
        return cost_columns[0]
    numeric_columns = df.select_dtypes(include=['float64', 'int64']).columns
    if len(numeric_columns) > 0:
        return numeric_columns[0]
    raise ValueError("Impossible d'identifier une colonne de coût dans les données")

nofeeder_cost_col = identify_cost_column(nofeeder_routing_costs)
feeder_cost_cols = [identify_cost_column(df) for df in feeders_routing_costs]

def cost_stats(df, cost_column):
    cost_data = df[cost_column].dropna()
    if len(cost_data) == 0:
        return None
    Q1 = cost_data.quantile(0.25)
    Q3 = cost_data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = cost_data[(cost_data < lower_bound) | (cost_data > upper_bound)]
    return {
        'count': len(cost_data),
        'mean': cost_data.mean(),
        'median': cost_data.median(),
        'std': cost_data.std(),
        'min': cost_data.min(),
        'max': cost_data.max(),
        'outliers(%)': len(outliers) / len(cost_data) * 100,
        'negatives': (cost_data < 0).sum(),
        'zeros': (cost_data == 0).sum()
    }

routing_stats = []
ref_stats = cost_stats(nofeeder_routing_costs, nofeeder_cost_col)
if ref_stats:
    ref_stats['scenario'] = "No Feeder"
    routing_stats.append(ref_stats)

for i, (feeder_routing_costs, cost_col) in enumerate(zip(feeders_routing_costs, feeder_cost_cols)):
    radius, frequency, speed, speed_index = extract_parameters(feeder_outputs[i])
    scenario_name = f"Feeder R{radius} F{frequency} S{speed:.1f}" if radius is not None else f"Feeder {i}"
    stats = cost_stats(feeder_routing_costs, cost_col)
    if stats:
        stats['scenario'] = scenario_name
        routing_stats.append(stats)

if routing_stats:
    df_routing_stats = pd.DataFrame(routing_stats).set_index('scenario')
    display(df_routing_stats.round(2))
else:
    print("Aucune statistique de coût n'a pu être calculée")

In [ ]:

merged_routing_costs = nofeeder_routing_costs.copy()
id_cols = ['person_id']
if 'trip_id' in merged_routing_costs.columns:
    id_cols.append('trip_id')
elif 'person_trip_id' in merged_routing_costs.columns:
    id_cols.append('person_trip_id')

cost_nofeeder = identify_cost_column(merged_routing_costs)
merged_routing_costs = merged_routing_costs.rename(columns={cost_nofeeder: 'routingCost_nofeeder'})

for i, feeder_df in enumerate(feeders_routing_costs):
    cost_feeder = identify_cost_column(feeder_df)
    merge_cols = [col for col in id_cols if col in feeder_df.columns]
    if not merge_cols:
        print(f"Feeder {i}: pas de colonnes communes")
        continue
    feeder_sub = feeder_df[merge_cols + [cost_feeder]].rename(columns={cost_feeder: f'routingCost_feeder_{i}'})
    merged_routing_costs = merged_routing_costs.merge(feeder_sub, on=merge_cols, how='left')

    diff_col = f'cost_difference_feeder_{i}'
    merged_routing_costs[diff_col] = merged_routing_costs['routingCost_nofeeder'] - merged_routing_costs[f'routingCost_feeder_{i}']
    diff = merged_routing_costs[diff_col].dropna()
    print(f"Feeder {i}: { (diff > 0).sum() } améliorations, { (diff < 0).sum() } dégradations")

print(f"Shape: {merged_routing_costs.shape}, Colonnes: {list(merged_routing_costs.columns)}")

In [ ]:
comparison_results = []

for i in range(len(feeders_routing_costs)):
    scenario_name = f"Feeder {i}"
    try:
        cost_nofeeder = merged_routing_costs['routingCost_nofeeder']
        cost_feeder = merged_routing_costs[f'routingCost_feeder_{i}']
        diff = merged_routing_costs[f'cost_difference_feeder_{i}']
        
        valid_mask = diff.notna()
        valid_nofeeder = cost_nofeeder[valid_mask]
        valid_diff = diff[valid_mask]
        valid_feeder = cost_feeder[valid_mask]

        improved = (valid_diff > 0).sum()
        worsened = (valid_diff < 0).sum()
        unchanged = (valid_diff == 0).sum()

        comparison_results.append({
        'scenario': scenario_name,
        'trips': len(valid_diff),
        'trips_improved': improved,
        'trips_worsened': worsened,
        'trips_unchanged': unchanged,
        'delta_mean_total': valid_diff.mean(),
        'delta_mean_diff': valid_diff[valid_diff != 0].mean(),
        'delta_total': valid_diff.sum(),
        'percent_improved': (improved / len(valid_diff)) * 100 if len(valid_diff) > 0 else 0
        })

    except Exception as e:
        print(f"{scenario_name}: Erreur - {e}")

if comparison_results:
    df_comparisons = pd.DataFrame(comparison_results)
    print(df_comparisons.round(2))
else:
    print("Aucune comparaison possible.")

In [ ]:
if (
    'df_comparisons' in locals()
    and not df_comparisons.empty
    and 'scenario' in df_comparisons.columns
    and 'df_routing_stats' in locals()
):
    grouped = list(df_comparisons.groupby("scenario"))
    n = len(grouped)
    scenarios_names = df_routing_stats.index.tolist()[1:]

    for batch_start in range(0, n, 4):
        batch = grouped[batch_start:batch_start + 4]
        rows = math.ceil(len(batch) / 2)

        subplot_titles = [
            scenarios_names[batch_start + i]
            if batch_start + i < len(scenarios_names)
            else f"Scénario {batch_start + i + 1}"
            for i in range(len(batch))
        ]

        fig = make_subplots(
            rows=rows,
            cols=2,
            specs=[[{'type': 'domain'}, {'type': 'domain'}] for _ in range(rows)],
            subplot_titles=subplot_titles,
        )


        for idx, (scenario, df_scenario) in enumerate(batch):
            row = idx // 2 + 1
            col = idx % 2 + 1
            total_improved = df_scenario['trips_improved'].sum()
            total_worsened = df_scenario['trips_worsened'].sum()
            total_unchanged = df_scenario['trips_unchanged'].sum()

            fig.add_trace(go.Pie(
                labels=['Améliorés', 'Dégradés', 'Inchangés'],
                values=[total_improved, total_worsened, total_unchanged],
                marker_colors=['green', 'red', 'gray'],
                textinfo='percent',
                textposition='inside',
                showlegend=(idx == 0),
                name="Répartition",
            ), row=row, col=col)

        fig.update_layout(
            title_text=f"Répartition des changements de coût (Scénarios {batch_start + 1} à {min(batch_start + 4, n)})",
            height=400 * rows,
            showlegend=True,
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=0,
                xanchor="center",
                x=0.5,
                font=dict(size=14)
            ),
            margin=dict(t=60, b=60),
            font=dict(size=14)
        )

        fig.show()
else:
    print("Données insuffisantes pour la comparaison.")

In [ ]:
cost_changes_data = []

for i, feeder_routing_costs in enumerate(feeders_routing_costs):
    radius, frequency, speed, speed_index = extract_parameters(feeder_outputs[i])
    scenario_name = f"R{radius} F{frequency} S{speed:.1f}" if radius is not None else f"Feeder {i}"
    
    nofeeder_cost_col = identify_cost_column(nofeeder_routing_costs)
    feeder_cost_col = identify_cost_column(feeder_routing_costs)
    
    merge_cols = ['person_id']
    if 'trip_id' in nofeeder_routing_costs.columns and 'trip_id' in feeder_routing_costs.columns:
        merge_cols.append('trip_id')
    elif 'person_trip_id' in nofeeder_routing_costs.columns and 'person_trip_id' in feeder_routing_costs.columns:
        merge_cols.append('person_trip_id')

    try:
        merged_costs = nofeeder_routing_costs[merge_cols + [nofeeder_cost_col]].merge(
            feeder_routing_costs[merge_cols + [feeder_cost_col]],
            on=merge_cols,
            how='inner',
            suffixes=('_ref', '_feeder')
        )
        
        cost_ref_col = f"{nofeeder_cost_col}_ref" if f"{nofeeder_cost_col}_ref" in merged_costs.columns else nofeeder_cost_col
        cost_feeder_col = f"{feeder_cost_col}_feeder" if f"{feeder_cost_col}_feeder" in merged_costs.columns else feeder_cost_col
        
        cost_diff = merged_costs[cost_feeder_col] - merged_costs[cost_ref_col]
        
        for diff in cost_diff:
            cost_changes_data.append({
                'scenario': scenario_name,
                'cost_change': diff,
                'radius': radius,
                'frequency': frequency,
                'speed': speed
            })
    except Exception as e:
        print(f"Erreur lors du calcul pour le scénario {i}: {e}")

if cost_changes_data:
    df_cost_changes = pd.DataFrame(cost_changes_data)

    fig_cost_changes = px.box(
        df_cost_changes,
        x='scenario',
        y='cost_change',
        title='Changements de Coût',
        labels={'cost_change': 'Changement de Coût', 'scenario': 'Scénario'},
        points='outliers'
    )
    fig_cost_changes.update_xaxes(tickangle=45)
    fig_cost_changes.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Pas de changement")
    fig_cost_changes.show()
else:
    print("Aucune donnée de changement de coût disponible pour la visualisation.")


In [ ]:
df_cost_changes_filtered = df_cost_changes[df_cost_changes["cost_change"] != 0]

fig_violin = px.violin(
    df_cost_changes_filtered,
    y="cost_change",
    x="scenario",
    box=True,
    points="outliers",
    color="scenario",
    title="Distribution détaillée des changements de coût (trajets modifiés uniquement)",
    labels={"cost_change": "Changement de coût", "scenario": "Scénario"}
)

fig_violin.update_layout(
    yaxis_title="NoFeeder - Feeder",
    xaxis_title="Scénario",
    showlegend=False,
    height=600
)
fig_violin.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="Pas de changement")

fig_violin.show()


In [ ]:
plot_data = []

for i, feeder_output in enumerate(feeder_outputs):
    radius, frequency, speed, speed_index = extract_parameters(feeder_output)
    
    if radius is not None:
        # Calculer le coût total moyen pour ce scénario
        routing_cost_col = f"routingCost_feeder_{i}"
        
        if routing_cost_col in merged_routing_costs.columns:
            total_cost = merged_routing_costs[routing_cost_col].mean()
            
            plot_data.append({
                'scenario_index': i,
                'radius': radius,
                'frequency': frequency,
                'speed': speed,
                'speed_index': speed_index,
                'total_cost': total_cost
            })

df_plot = pd.DataFrame(plot_data)
print("Données pour les graphiques:")
print(df_plot)

In [ ]:
fig = go.Figure()

frequencies = sorted(df_plot['frequency'].unique())
speeds = sorted(df_plot['speed'].unique())

colors = px.colors.qualitative.Set1[:len(frequencies)]
frequency_colors = {freq: colors[i] for i, freq in enumerate(frequencies)}

line_styles = ['solid', 'dash', 'dot', 'dashdot']
speed_styles = {speed: line_styles[i % len(line_styles)] for i, speed in enumerate(speeds)}

for freq in frequencies:
    for speed in speeds:
        mask = (df_plot['frequency'] == freq) & (df_plot['speed'] == speed)
        subset = df_plot[mask].sort_values('radius')
        
        if len(subset) > 0:
            line_name = f"Freq: {freq}, Speed: {speed}"
            
            fig.add_trace(go.Scatter(
                x=subset['radius'],
                y=subset['total_cost'],
                mode='lines+markers',
                name=line_name,
                line=dict(
                    color=frequency_colors[freq],
                    dash=speed_styles[speed],
                    width=2
                ),
                marker=dict(
                    size=6,
                    color=frequency_colors[freq]
                )
            ))

fig.update_layout(
    title='Coût Total de Routing vs Rayon du Feeder',
    xaxis_title='Rayon du Feeder (m)',
    yaxis_title='Coût Total Moyen de Routing',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.01
    ),
    width=800,
    height=500
)

fig.add_annotation(
    x=0.02, y=0.98,
    xref="paper", yref="paper",
    text="Couleurs = Fréquences<br>Styles de ligne = Vitesses",
    showarrow=False,
    font=dict(size=10),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="black",
    borderwidth=1
)

fig.show()


In [ ]:
improvement_data = []

for i, feeder_output in enumerate(feeder_outputs):
    radius, frequency, speed, speed_index = extract_parameters(feeder_output)
    
    if radius is not None:
        cost_diff_col = f"cost_difference_feeder_{i}"
        
        if cost_diff_col in merged_routing_costs.columns:
            improved_trips = (merged_routing_costs[cost_diff_col] > 0).sum()
            total_trips = len(merged_routing_costs[cost_diff_col].dropna())
            improvement_rate = (improved_trips / total_trips) * 100 if total_trips > 0 else 0
            
            improvement_data.append({
                'scenario_index': i,
                'radius': radius,
                'frequency': frequency,
                'speed': speed,
                'speed_index': speed_index,
                'improved_trips': improved_trips,
                'total_trips': total_trips,
                'improvement_rate': improvement_rate
            })

df_improvement = pd.DataFrame(improvement_data)
print("Données d'amélioration des trips:")
print(df_improvement)


In [ ]:
fig2 = go.Figure()

frequencies = sorted(df_improvement['frequency'].unique())
speeds = sorted(df_improvement['speed'].unique())

colors = px.colors.qualitative.Set1[:len(frequencies)]
frequency_colors = {freq: colors[i] for i, freq in enumerate(frequencies)}

line_styles = ['solid', 'dash', 'dot', 'dashdot']
speed_styles = {speed: line_styles[i % len(line_styles)] for i, speed in enumerate(speeds)}

for freq in frequencies:
    for speed in speeds:
        mask = (df_improvement['frequency'] == freq) & (df_improvement['speed'] == speed)
        subset = df_improvement[mask].sort_values('radius')
        
        if len(subset) > 0:
            line_name = f"Freq: {freq}, Speed: {speed}"
            
            fig2.add_trace(go.Scatter(
                x=subset['radius'],
                y=subset['improved_trips'],
                mode='lines+markers',
                name=line_name,
                line=dict(
                    color=frequency_colors[freq],
                    dash=speed_styles[speed],
                    width=2
                ),
                marker=dict(
                    size=6,
                    color=frequency_colors[freq]
                ),
                hovertemplate=f'<b>{line_name}</b><br>' +
                             'Rayon: %{x}m<br>' +
                             'Trips améliorés: %{y}<br>' +
                             'Taux d\'amélioration: %{customdata:.1f}%<extra></extra>',
                customdata=subset['improvement_rate']
            ))

fig2.update_layout(
    title='Nombre de Trips avec Coût Amélioré vs Rayon du Feeder',
    xaxis_title='Rayon du Feeder (m)',
    yaxis_title='Nombre de Trips Améliorés',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.01
    ),
    width=800,
    height=500
)

fig2.add_annotation(
    x=0.02, y=0.98,
    xref="paper", yref="paper",
    text="Couleurs = Fréquences<br>Styles de ligne = Vitesses",
    showarrow=False,
    font=dict(size=10),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="black",
    borderwidth=1
)

fig2.show()

In [ ]:
fig3 = go.Figure()

for freq in frequencies:
    for speed in speeds:
        mask = (df_improvement['frequency'] == freq) & (df_improvement['speed'] == speed)
        subset = df_improvement[mask].sort_values('radius')
        
        if len(subset) > 0:
            line_name = f"Freq: {freq}, Speed: {speed}"
            
            fig3.add_trace(go.Scatter(
                x=subset['radius'],
                y=subset['improvement_rate'],
                mode='lines+markers',
                name=line_name,
                line=dict(
                    color=frequency_colors[freq],
                    dash=speed_styles[speed],
                    width=2
                ),
                marker=dict(
                    size=6,
                    color=frequency_colors[freq]
                ),
                hovertemplate=f'<b>{line_name}</b><br>' +
                             'Rayon: %{x}m<br>' +
                             'Taux d\'amélioration: %{y:.1f}%<br>' +
                             'Trips améliorés: %{customdata}<extra></extra>',
                customdata=subset['improved_trips']
            ))

fig3.update_layout(
    title='Taux d\'Amélioration des Trips (%) vs Rayon du Feeder',
    xaxis_title='Rayon du Feeder (m)',
    yaxis_title='Taux d\'Amélioration (%)',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.01
    ),
    width=800,
    height=500
)

fig3.add_annotation(
    x=0.02, y=0.98,
    xref="paper", yref="paper",
    text="Couleurs = Fréquences<br>Styles de ligne = Vitesses",
    showarrow=False,
    font=dict(size=10),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="black",
    borderwidth=1
)

fig3.show()


In [ ]:
cost_components_data = []

for i, feeder_routing_costs in enumerate(feeders_routing_costs):
    radius, frequency, speed, speed_index = extract_parameters(feeder_outputs[i])
    
    if radius is not None:
        
        cost_cols = identify_cost_column(feeder_routing_costs)
        if isinstance(cost_cols, str):
            cost_cols = [cost_cols]

        
        avg_costs = feeder_routing_costs[cost_cols].mean()

        if isinstance(avg_costs, (float, np.floating)):
            avg_costs = pd.Series({cost_cols[0]: avg_costs})
        
        for component, value in avg_costs.items():
            cost_components_data.append({
                'scenario_index': i,
                'radius': radius,
                'frequency': frequency,
                'speed': speed,
                'speed_index': speed_index,
                'cost_component': component,
                'cost_value': value
            })

df_cost_components = pd.DataFrame(cost_components_data)
print("\nDonnées des coûts:")
print(df_cost_components.head(10))

In [ ]:
pt_usage_data = []

#nofeeder
nofeeder_pt_modes = nofeeder_pt['transit_mode'].value_counts()
for mode, count in nofeeder_pt_modes.items():
    pt_usage_data.append({
        'scenario': 'No Feeder',
        'radius': 0,
        'frequency': 0,
        'speed': 0,
        'pt_mode': mode,
        'leg_count': count,
        'scenario_type': 'baseline'
    })

# scénarios avec feeder
for i, feeder_pt in enumerate(feeders_pt):
    radius, frequency, speed, speed_index = extract_parameters(feeder_outputs[i])
    
    if radius is not None:
        feeder_pt_modes = feeder_pt['transit_mode'].value_counts()
        
        for mode, count in feeder_pt_modes.items():
            pt_usage_data.append({
                'scenario': f'Feeder R{radius} F{frequency} S{speed:.1f}',
                'radius': radius,
                'frequency': frequency,
                'speed': speed,
                'pt_mode': mode,
                'leg_count': count,
                'scenario_type': 'feeder'
            })

df_pt_usage = pd.DataFrame(pt_usage_data)
print("Données d'usage PT:")
print(df_pt_usage.head(10))

In [ ]:
subset = df_pt_usage[df_pt_usage['scenario_type'] == 'feeder']
grouped = subset.groupby(['frequency', 'speed', 'radius'])

pie_data = []
for (freq, speed, radius), group in grouped:
    pie_data.append({
        'frequency': freq,
        'speed': speed,
        'radius': radius,
        'data': group
    })

max_per_fig = 4
n_figs = math.ceil(len(pie_data) / max_per_fig)

for fig_index in range(n_figs):
    start = fig_index * max_per_fig
    end = start + max_per_fig
    current_pies = pie_data[start:end]

    n_charts = len(current_pies)
    n_cols = 2
    n_rows = math.ceil(n_charts / n_cols)

    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        specs=[[{'type': 'domain'}]*n_cols for _ in range(n_rows)],
        subplot_titles=[
            f"R={entry['radius']}m | F={entry['frequency']} | V={entry['speed']}"
            for entry in current_pies
        ]
    )

    for i, entry in enumerate(current_pies):
        row = i // n_cols + 1
        col = i % n_cols + 1
        df = entry['data']
        fig.add_trace(
            go.Pie(
                labels=df['pt_mode'],
                values=df['leg_count'],
                name=f"R={entry['radius']}, F={entry['frequency']}, V={entry['speed']}",
                textinfo='percent',
                showlegend=True
            ),
            row=row, col=col
        )

    fig.update_layout(
        title_text=f"Répartition des modes de transport en commun ({start + 1} à {min(end, len(pie_data))})",
        height=350 * n_rows,
        width=900,
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02
        )
    )

    fig.show()

In [ ]:
baseline_counts = df_pt_usage[df_pt_usage['scenario_type'] == 'baseline'].set_index('pt_mode')['leg_count']

comparison_data = []
for _, row in df_pt_usage[df_pt_usage['scenario_type'] == 'feeder'].iterrows():
    baseline_count = baseline_counts.get(row['pt_mode'], 0)
    difference = row['leg_count'] - baseline_count
    percentage_change = (difference / baseline_count * 100) if baseline_count > 0 else float('inf')
    
    comparison_data.append({
        'radius': row['radius'],
        'frequency': row['frequency'],
        'speed': row['speed'],
        'pt_mode': row['pt_mode'],
        'difference': difference,
        'percentage_change': percentage_change if percentage_change != float('inf') else 0
    })

df_comparison = pd.DataFrame(comparison_data)

fig6 = px.bar(
    df_comparison,
    x='radius',
    y='percentage_change',
    color='pt_mode',
    facet_col='frequency',
    facet_row='speed',
    title='Changement d\'usage des transports en commun (%) par rapport au scénario sans feeder',
    labels={
        'percentage_change': 'Changement (%)',
        'radius': 'Rayon du Feeder (m)',
        'pt_mode': 'Mode PT'
    },
    height=600,
    width=1000
)

fig6.add_hline(y=0, line_dash="dash", line_color="black")

fig6.update_layout(
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig6.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig6.show()